In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEvent
from pathlib import Path
#invite people for the Kaggle party
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
# from scipy import stats
import warnings
import time
warnings.filterwarnings('ignore')

In [ ]:
### cell 0 ###

benchmark_name = "comprehensive-data-exploration-with-python"
df_train = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "train.csv"
)
factor = 3
# replicate the entire DataFrame in blocks (GPU‐friendly concat)
df_train = pd.concat([df_train] * factor, ignore_index=True)
df_train.info()

In [ ]:
### cell 1 ###

#check the decoration
df_train.columns

In [ ]:
### cell 2 ###

#descriptive statistics summary
df_train['SalePrice'].describe()

In [ ]:
### cell 3 ###

#skewness and kurtosis
print("Skewness: %f" % df_train['SalePrice'].skew())
print("Kurtosis: %f" % df_train['SalePrice'].kurt())

In [ ]:
### cell 4 ###

var = 'GrLivArea'
data = df_train[['SalePrice', var]]

In [ ]:
### cell 5 ###

var = 'TotalBsmtSF'
# Directly select both columns to avoid concat overhead
data = df_train[['SalePrice', var]]

In [ ]:
### cell 6 ###

var = 'OverallQual'
data = df_train[['SalePrice', var]]

In [ ]:
### cell 7 ###

var = 'YearBuilt'
# Select both columns in one go instead of two getitem calls + concat
data = df_train[['SalePrice', var]]

In [ ]:
### cell 8 ###

numeric_df = df_train.select_dtypes(include=["number"])
corrmat = numeric_df.corr()

In [ ]:
### cell 9 ###

#saleprice correlation matrix
k = 10 #number of variables for heatmap
cols = corrmat.sort_values(by='SalePrice', ascending=False).head(k)['SalePrice'].index


cm = np.corrcoef(df_train[cols].values.T)

In [ ]:
### cell 10 ###

#missing data
total = df_train.isnull().sum().sort_values(ascending=False)
percent = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
### cell 11 ###

# Drop any column with more than one missing value in one GPU‐accelerated pass
df_train = df_train.dropna(axis=1, thresh=len(df_train) - 1)

# just checking that there's no missing data left
df_train.isnull().sum().max()

In [ ]:
### cell 12 ###

#standardizing data
saleprice_scaled = StandardScaler().fit_transform(df_train['SalePrice'].to_numpy()[:, np.newaxis])
low_range = saleprice_scaled[saleprice_scaled[:,0].argsort()][:10]
high_range= saleprice_scaled[saleprice_scaled[:,0].argsort()][-10:]
print(low_range)
print(high_range)

In [ ]:
### cell 13 ###

# bivariate analysis saleprice/GrLivArea
var = 'GrLivArea'
data = df_train[['SalePrice', var]]

In [ ]:
### cell 14 ###

# deleting points
# Display the two records with the largest GrLivArea
# (uses a single GPU sort + head)
df_train.sort_values('GrLivArea', ascending=False).head(2)

# Remove both outliers in one vectorized operation
df_train = df_train[~df_train['Id'].isin([1299, 524])]


In [ ]:
### cell 15 ###

var = 'TotalBsmtSF'
data = df_train[['SalePrice', var]]

In [ ]:
### cell 16 ###

# Applying log transformation using numpy's ufunc (dispatches to GPU under cudf.pandas)
df_train['SalePrice'] = np.log(df_train['SalePrice'])

In [ ]:
### cell 17 ###

# data transformation
# revert to the NumPy ufunc, which dispatches to the GPU via __array_ufunc__
df_train['GrLivArea'] = np.log(df_train['GrLivArea'])

In [ ]:
### cell 18 ###

#create column for new variable (one is enough because it's a binary categorical feature)
#if area>0 it gets 1, for area==0 it gets 0
df_train['HasBsmt'] = pd.Series(len(df_train['TotalBsmtSF']), index=df_train.index)
df_train['HasBsmt'] = 0 
df_train.loc[df_train['TotalBsmtSF']>0,'HasBsmt'] = 1

In [ ]:
### cell 19 ###

# transform data: apply log only where there is a basement
# use GPU‐accelerated np.log via the __array_ufunc__ override
df_train['TotalBsmtSF'] = df_train['TotalBsmtSF'].mask(
    df_train['HasBsmt'] == 1,
    np.log(df_train['TotalBsmtSF'])
)

In [ ]:
### cell 20 ###

# histogram and normal probability plot
bsmt_sf = df_train['TotalBsmtSF']              # grab the column once
pos_mask = bsmt_sf > 0                       # compute mask once
positive_bsmt_sf = bsmt_sf[pos_mask]          # filter once
_ = positive_bsmt_sf                         # first use
_ = positive_bsmt_sf                         # second use

In [ ]:
### cell 21 ###

# scatter plot
filtered = df_train[df_train['TotalBsmtSF'] > 0]
_ = filtered['TotalBsmtSF']
_ = filtered['SalePrice']

In [ ]:
### cell 22 ###

#convert categorical variable into dummy
df_train = pd.get_dummies(df_train)